# Atividade: Extração de Múltiplas Fontes e Data Wrangling

**Nome(s):**  
**Data:** 

---

### Roteiro

| Parte | Tópico |
|-------|--------|
| Parte 1 | Configuração e extração da API REST (IBGE) |
| Parte 2 | Extração de arquivo CSV (GitHub) e profiling inicial |
| Parte 3 | Diagnóstico de qualidade e aplicação de Data Wrangling |
| Parte 4 | Integração das fontes e validação final |
| Parte 5 | Análises exploratórias e reflexão |


#### Regra:

> **Escreva todo o código por conta própria**. Consulte a documentação das bibliotecas sempre que necessário.

---
## Contexto de Negócio

Você trabalha como **Engenheiro(a) de Dados** em uma startup de inteligência de
mercado imobiliário. A empresa quer lançar uma plataforma que ajude investidores
a comparar o potencial de diferentes municípios brasileiros.

Para isso, você precisa construir uma **base de dados integrada de municípios**,
combinando informações geográficas, administrativas e demográficas provenientes
de **duas fontes distintas**:

| Fonte | Tipo | Dados disponíveis |
|-------|------|-------------------|
| **API IBGE Localidades** | REST API (JSON) | Hierarquia geográfica, mesorregião, microrregião, UF, região |
| **CSV de Municípios Brasileiros** | Arquivo CSV (GitHub) | Coordenadas geográficas, indicador de capital, nomes e códigos de subdivisões |

Ao combinar as duas fontes, você descobrirá **inconsistências reais** entre elas e deverá aplicar técnicas de **Data Wrangling** para criar uma base integrada
e confiável.


---
## Fontes de Dados

### Fonte 1 - API IBGE Localidades (REST/JSON)

O IBGE disponibiliza uma API pública e gratuita sem necessidade de cadastro ou autenticação.
Documentação completa: **https://servicodados.ibge.gov.br/api/docs/localidades**

Endpoints que você utilizará:

| Endpoint | URL | Retorna |
|----------|-----|---------|
| Municípios | `https://servicodados.ibge.gov.br/api/v1/localidades/municipios` | Lista de todos os 5.570 municípios |
| Estados | `https://servicodados.ibge.gov.br/api/v1/localidades/estados` | 27 UFs com região |
| Regiões | `https://servicodados.ibge.gov.br/api/v1/localidades/regioes` | 5 regiões do Brasil |

A resposta do endpoint de municípios é um **array JSON** com objetos aninhados no formato:
```json
{
  "id": 5002704,
  "nome": "Água Clara",
  "microrregiao": {
    "id": 50005,
    "nome": "Três Lagoas",
    "mesorregiao": {
      "id": 5001,
      "nome": "Leste de Mato Grosso do Sul",
      "UF": {
        "id": 50,
        "sigla": "MS",
        "nome": "Mato Grosso do Sul",
        "regiao": { "id": 5, "sigla": "CO", "nome": "Centro-Oeste" }
      }
    }
  }
}
```
> **Atenção:** A resposta é um JSON **aninhado** (nested). Você precisará *achatar* (flatten)
> essa estrutura para carregá-la em um DataFrame tabular.

---

### Fonte 2 - CSV de Municípios Brasileiros (GitHub)

URL direta para download:
```
https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv
```

Colunas disponíveis no arquivo:

| Coluna | Descrição |
|--------|----------|
| `codigo_ibge` | Código IBGE do município (7 dígitos) |
| `nome` | Nome do município |
| `latitude` | Latitude do centroide |
| `longitude` | Longitude do centroide |
| `capital` | 1 se é capital estadual, 0 caso contrário |
| `codigo_uf` | Código numérico da UF |
| `codigo_mesorregiao` | Código da mesorregião |
| `nome_mesorregiao` | Nome da mesorregião |
| `codigo_microrregiao` | Código da microrregião |
| `nome_microrregiao` | Nome da microrregião |


---
# Parte 1 - Configuração e Extração via API REST

### Tarefa 1.1 - Instalação e importações

Instale e importe as bibliotecas necessárias para a atividade.  
Você precisará de: `requests` (requisições HTTP), `pandas`, `numpy`, `json`.


In [1]:
# Tarefa 1.1 - Instale e importe as bibliotecas necessárias
import numpy as np
import requests
import pandas as pd
import json
import unidecode

### Tarefa 1.2 - Extrair dados dos municípios via API

Faça uma requisição GET ao endpoint de municípios do IBGE e armazene a resposta.  
Verifique o status code da resposta antes de prosseguir.

> **Dica:** Use `requests.get(url)` e verifique se `response.status_code == 200`.  
> O conteúdo pode ser convertido para lista Python com `response.json()`.


In [2]:
# Tarefa 1.2 - Faça a requisição à API e inspecione a resposta
municipios = requests.get("https://servicodados.ibge.gov.br/api/v1/localidades/municipios")
print(municipios.status_code)  # Verificar se a requisição foi bem-sucedida
estados = requests.get("https://servicodados.ibge.gov.br/api/v1/localidades/estados")
print(estados.status_code)  # Verificar se a requisição foi bem-sucedida
regioes = requests.get("https://servicodados.ibge.gov.br/api/v1/localidades/regioes")
print(regioes.status_code)  # Verificar se a requisição foi bem-sucedida

200
200
200


### Tarefa 1.3 - Achatar o JSON aninhado (Flatten)

A resposta da API é um JSON aninhado com vários níveis. Você precisa transformá-lo
em uma estrutura tabular (um registro por município) com as seguintes colunas:

| Coluna desejada | Caminho no JSON |
|-----------------|----------------|
| `id_municipio` | `id` |
| `nome_municipio` | `nome` |
| `id_microrregiao` | `microrregiao.id` |
| `nome_microrregiao` | `microrregiao.nome` |
| `id_mesorregiao` | `microrregiao.mesorregiao.id` |
| `nome_mesorregiao` | `microrregiao.mesorregiao.nome` |
| `sigla_uf` | `microrregiao.mesorregiao.UF.sigla` |
| `nome_uf` | `microrregiao.mesorregiao.UF.nome` |
| `id_uf` | `microrregiao.mesorregiao.UF.id` |
| `sigla_regiao` | `microrregiao.mesorregiao.UF.regiao.sigla` |
| `nome_regiao` | `microrregiao.mesorregiao.UF.regiao.nome` |


> **Dica 1:** Itere pela lista de municípios e extraia cada campo do dicionário aninhado.  
> **Dica 2:** A função `pd.json_normalize()` pode simplificar esse processo para objetos simples.


In [3]:
# Tarefa 1.3 - Achate o JSON e crie um DataFrame com as colunas solicitadas
municipios = municipios.json()
df_municipios = pd.json_normalize(municipios)

regioes = regioes.json()
df_regioes = pd.json_normalize(regioes)

estados = estados.json()
df_estados = pd.json_normalize(estados)


### Tarefa 1.4 - Verificação inicial da extração

Após criar o DataFrame, responda:
- Quantos municípios foram extraídos? (o Brasil possui 5.570)
- Quais são os tipos de cada coluna?
- Existem valores nulos?
- Exiba as primeiras 5 linhas do DataFrame.


In [4]:
# Tarefa 1.4 - Verifique a integridade do DataFrame extraído da API
print(df_municipios.info())

print(df_regioes.info())

print(df_estados.info())

<class 'pandas.DataFrame'>
RangeIndex: 5571 entries, 0 to 5570
Data columns (total 23 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   id                                                    5571 non-null   int64  
 1   nome                                                  5571 non-null   str    
 2   microrregiao.id                                       5570 non-null   float64
 3   microrregiao.nome                                     5570 non-null   str    
 4   microrregiao.mesorregiao.id                           5570 non-null   float64
 5   microrregiao.mesorregiao.nome                         5570 non-null   str    
 6   microrregiao.mesorregiao.UF.id                        5570 non-null   float64
 7   microrregiao.mesorregiao.UF.sigla                     5570 non-null   str    
 8   microrregiao.mesorregiao.UF.nome                      5570 non-null  

### Colunas provenientes da microregiao com um dado faltante

---
# Parte 2 - Extração do CSV e Data Profiling

### Tarefa 2.1 - Extrair o CSV diretamente da URL

Carregue o arquivo CSV de municípios brasileiros diretamente da URL do GitHub
em um DataFrame.

> **Dica:** O `pandas.read_csv()` aceita URLs diretamente como argumento.


In [5]:
# Tarefa 2.1 - Carregue o CSV da URL diretamente para um DataFrame
dataset_url = "https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv"
df_municipios_csv = pd.read_csv(dataset_url, sep=",")
df_municipios_csv.info()

<class 'pandas.DataFrame'>
RangeIndex: 5571 entries, 0 to 5570
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   codigo_ibge   5571 non-null   int64  
 1   nome          5571 non-null   str    
 2   latitude      5571 non-null   float64
 3   longitude     5571 non-null   float64
 4   capital       5571 non-null   int64  
 5   codigo_uf     5571 non-null   int64  
 6   siafi_id      5571 non-null   int64  
 7   ddd           5571 non-null   int64  
 8   fuso_horario  5571 non-null   str    
dtypes: float64(2), int64(5), str(2)
memory usage: 391.8 KB


### Tarefa 2.2 - Data Profiling do CSV

Realize um profiling completo do DataFrame do CSV. Para cada coluna, levante:

- Tipo de dado atual
- Quantidade e percentual de valores nulos
- Quantidade de valores únicos (cardinalidade)
- Para colunas numéricas: mínimo, máximo, média e desvio padrão
- Para colunas categóricas/texto: os 5 valores mais frequentes
- Existência de valores duplicados por código IBGE


In [6]:
# Tarefa 2.2a - Análise de nulos e cardinalidade de cada coluna
# print(df_municipios_csv.isnull().sum())
print(df_municipios_csv.nunique())

codigo_ibge     5571
nome            5300
latitude        5527
longitude       5477
capital            2
codigo_uf         27
siafi_id        5571
ddd               67
fuso_horario       4
dtype: int64


In [7]:
# Tarefa 2.2b - Estatísticas descritivas das colunas numéricas
df_municipios_csv.describe()

,codigo_ibge,latitude,longitude,capital,codigo_uf,siafi_id,ddd
count,5.571000e+03,5571.000000,5571.000000,5571.000000,5571.000000,5571.000000,5571.000000
mean,3.253923e+06,-16.448572,-46.232613,0.004847,32.381081,4519.279124,57.101059
std,9.851332e+05,8.286624,6.409116,0.069454,9.836144,3050.210088,25.420747
min,1.100015e+06,-33.686600,-72.899700,0.000000,11.000000,1.000000,11.000000
25%,2.512150e+06,-22.843550,-50.881400,0.000000,25.000000,1594.000000,35.000000
50%,3.146305e+06,-18.092100,-46.525100,0.000000,31.000000,4381.000000,55.000000
75%,4.119226e+06,-8.496480,-41.410950,0.000000,41.000000,7180.000000,82.000000
max,5.300108e+06,4.603140,-32.410700,1.000000,53.000000,9997.000000,99.000000


In [8]:
# Tarefa 2.2c - Distribuição de frequência das colunas categóricas (top 5 valores)
df_municipios_csv["capital"] = df_municipios_csv["capital"].astype(bool)
df_municipios_csv["codigo_uf"] = df_municipios_csv["codigo_uf"].astype(str)
df_municipios_csv["ddd"] = df_municipios_csv["ddd"].astype(str)
print(df_municipios_csv["nome"].value_counts().head(5))
print(df_municipios_csv["capital"].value_counts().head(5))
print(df_municipios_csv["codigo_uf"].value_counts().head(5))
print(df_municipios_csv["ddd"].value_counts().head(5))


nome
Bom Jesus       5
São Domingos    5
Bonito          4
Planalto        4
Santa Helena    4
Name: count, dtype: int64
capital
False    5544
True       27
Name: count, dtype: int64
codigo_uf
31    853
35    645
43    497
29    417
41    399
Name: count, dtype: int64
ddd
83    223
33    175
84    167
55    160
35    159
Name: count, dtype: int64


In [9]:
# Tarefa 2.2d - Verifique duplicatas pelo código IBGE
df_municipios_csv["codigo_ibge"].duplicated().sum()

np.int64(0)

### Tarefa 2.3 - Data Profiling do DataFrame da API

Realize o mesmo profiling da Tarefa 2.2, agora para o DataFrame extraído da API IBGE.
Preste atenção especial às colunas de texto: elas virão com acentuação?
Os nomes de cidades estão padronizados?


In [10]:
# Tarefa 2.3 - Análise de nulos e cardinalidade de cada coluna
print(df_municipios.isnull().sum())
print(df_municipios.nunique())

print(df_regioes.isnull().sum())
print(df_regioes.nunique())

print(df_estados.isnull().sum())
print(df_estados.nunique())

id                                                         0
nome                                                       0
microrregiao.id                                            1
microrregiao.nome                                          1
microrregiao.mesorregiao.id                                1
microrregiao.mesorregiao.nome                              1
microrregiao.mesorregiao.UF.id                             1
microrregiao.mesorregiao.UF.sigla                          1
microrregiao.mesorregiao.UF.nome                           1
microrregiao.mesorregiao.UF.regiao.id                      1
microrregiao.mesorregiao.UF.regiao.sigla                   1
microrregiao.mesorregiao.UF.regiao.nome                    1
regiao-imediata.id                                         0
regiao-imediata.nome                                       0
regiao-imediata.regiao-intermediaria.id                    0
regiao-imediata.regiao-intermediaria.nome                  0
regiao-imediata.regiao-i

In [14]:
# - Estatísticas descritivas das colunas numéricas
print(df_municipios.describe())
print(df_regioes.describe())
print(df_estados.describe())

                 id  microrregiao.id  microrregiao.mesorregiao.id  \
count  5.571000e+03      5570.000000                  5570.000000   
mean   3.253923e+06     32395.371454                  3242.038779   
std    9.851332e+05      9834.924338                   983.637524   
min    1.100015e+06     11001.000000                  1101.000000   
25%    2.512150e+06     25015.000000                  2503.000000   
50%    3.146305e+06     31047.000000                  3110.000000   
75%    4.119226e+06     41024.000000                  4106.000000   
max    5.300108e+06     53001.000000                  5301.000000   

       microrregiao.mesorregiao.UF.id  microrregiao.mesorregiao.UF.regiao.id  \
count                     5570.000000                            5570.000000   
mean                        32.377738                               2.897846   
std                          9.833862                               1.088215   
min                         11.000000                     

In [12]:
# - Distribuição de frequência das colunas categóricas (top 5 valores)
print(df_municipios.select_dtypes(include=['str']).value_counts().head(5))
print(df_regioes.select_dtypes(include=['str']).value_counts().head(5))
print(df_estados.select_dtypes(include=['str']).value_counts().head(5))

nome                   microrregiao.nome  microrregiao.mesorregiao.nome  microrregiao.mesorregiao.UF.sigla  microrregiao.mesorregiao.UF.nome  microrregiao.mesorregiao.UF.regiao.sigla  microrregiao.mesorregiao.UF.regiao.nome  regiao-imediata.nome  regiao-imediata.regiao-intermediaria.nome  regiao-imediata.regiao-intermediaria.UF.sigla  regiao-imediata.regiao-intermediaria.UF.nome  regiao-imediata.regiao-intermediaria.UF.regiao.sigla  regiao-imediata.regiao-intermediaria.UF.regiao.nome
Alta Floresta D'Oeste  Cacoal             Leste Rondoniense              RO                                 Rondônia                          N                                         Norte                                    Cacoal                Ji-Paraná                                  RO                                             Rondônia                                      N                                                     Norte                                                  1
Ariquemes        

In [ ]:
# - Verifique duplicatas pelo código IBGE
print(df_municipios["id"].duplicated().sum())
print(df_estados["id"].duplicated().sum())
print(df_regioes["id"].duplicated().sum())


0
0
0


---
# Parte 3 - Diagnóstico de Qualidade e Data Wrangling

### Tarefa 3.1 - Diagnóstico comparativo entre as duas fontes

Compare os dois DataFrames para identificar possíveis inconsistências **entre** eles.  
Investigue:

1. Os **dois datasets possuem o mesmo número de municípios?** Se não, quais estão faltando em um ou outro?
2. Os **nomes dos municípios** são idênticos nas duas fontes, ou existem variações de grafia?
3. Os **nomes de mesorregiões e microrregiões** são consistentes entre as duas fontes?
4. O campo `codigo_ibge` do CSV corresponde ao campo `id_municipio` da API?
   Atenção: o CSV usa 7 dígitos e a API pode usar formato diferente.

> **Dica:** Faça um merge entre os dois DataFrames por código IBGE e analise os registros
> que não encontraram correspondência (`how='outer'` + verificar NaNs).


In [11]:
# Normalização de colunas para garantir consistência entre os DataFrames
import unicodedata

def normalizar(texto):
    if pd.isna(texto):
        return texto
        
    texto = texto.lower().strip()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    
    return texto

In [12]:
# Tarefa 3.1a - Compare o volume de municípios nas duas fontes

df_municipios_csv.shape[0], df_municipios.shape[0]


(5571, 5571)

In [16]:
print("CSV shape:", df_municipios_csv.shape)
print("API shape:", df_municipios.shape)

CSV shape: (5571, 9)
API shape: (5571, 23)


In [25]:
# Tarefa 3.1b - Identifique municípios presentes em uma fonte mas ausentes na outra

df_municipios_csv = df_municipios_csv.rename(columns={"codigo_ibge": "id"})

df_municipios_csv["nome"] = df_municipios_csv["nome"].apply(normalizar)
df_municipios["nome"] = df_municipios["nome"].apply(normalizar)

df_merge = df_municipios_csv.merge(df_municipios, on="id", suffixes=("_csv", "_municipios"))
diferentes = df_merge[df_merge["nome_csv"] != df_merge["nome_municipios"]]
print(diferentes[["id", "nome_csv", "nome_municipios"]])

correcoes = dict(zip(diferentes["nome_csv"], diferentes["nome_municipios"]))
df_merge["nome_municipios"] = df_merge["nome_municipios"].replace(correcoes)


           id                       nome_csv            nome_municipios
207   2800100        amparo de sao francisco    amparo do sao francisco
427   2401305  augusto severo (campo grande)               campo grande
493   3105509            barao de monte alto        barao do monte alto
628   3506607                 biritiba-mirim             biritiba mirim
1608  3122900                   dona eusebia               dona euzebia
1815  3516101                       florinia                   florinea
1833  1708254           fortaleza do tabocao                    tabocao
1956  2802601                 gracho cardoso            graccho cardoso
1966  4206108                      grao para                  grao-para
2519  2405306     januario cicco (boa saude)             januario cicco
3220  2922250        muquem de sao francisco    muquem do sao francisco
3441  2408409          olho-d'agua do borges      olho d'agua do borges
3442  3145455                   olhos d'agua               olhos

In [24]:
df_merge.head()

,id,nome_csv,latitude,longitude,capital,codigo_uf,siafi_id,ddd,fuso_horario,nome_municipios,...,regiao-imediata.nome,regiao-imediata.regiao-intermediaria.id,regiao-imediata.regiao-intermediaria.nome,regiao-imediata.regiao-intermediaria.UF.id,regiao-imediata.regiao-intermediaria.UF.sigla,regiao-imediata.regiao-intermediaria.UF.nome,regiao-imediata.regiao-intermediaria.UF.regiao.id,regiao-imediata.regiao-intermediaria.UF.regiao.sigla,regiao-imediata.regiao-intermediaria.UF.regiao.nome,microrregiao
0,5200050,abadia de goias,-16.75730,-49.4412,False,52,1050,62,America/Sao_Paulo,abadia de goias,...,Goiânia,5201,Goiânia,52,GO,Goiás,5,CO,Centro-Oeste,NaN
1,3100104,abadia dos dourados,-18.48310,-47.3916,False,31,4001,34,America/Sao_Paulo,abadia dos dourados,...,Monte Carmelo,3111,Uberlândia,31,MG,Minas Gerais,3,SE,Sudeste,NaN
2,5200100,abadiania,-16.19700,-48.7057,False,52,9201,62,America/Sao_Paulo,abadiania,...,Anápolis,5201,Goiânia,52,GO,Goiás,5,CO,Centro-Oeste,NaN
3,3100203,abaete,-19.15510,-45.4444,False,31,4003,37,America/Sao_Paulo,abaete,...,Abaeté,3113,Divinópolis,31,MG,Minas Gerais,3,SE,Sudeste,NaN
4,1500107,abaetetuba,-1.72183,-48.8788,False,15,401,91,America/Sao_Paulo,abaetetuba,...,Abaetetuba,1501,Belém,15,PA,Pará,1,N,Norte,NaN


In [ ]:
# df_merge["nome_csv"] = df_merge["nome_csv"].replace(correcoes)
# df_merge = df_merge.drop(columns=["nome_municipios"])
# df_merge.rename(columns={"nome_csv": "nome"}, inplace=True)

df_merge.info()

<class 'pandas.DataFrame'>
RangeIndex: 5571 entries, 0 to 5570
Data columns (total 30 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   id                                                    5571 non-null   int64  
 1   nome                                                  5571 non-null   str    
 2   latitude                                              5571 non-null   float64
 3   longitude                                             5571 non-null   float64
 4   capital                                               5571 non-null   bool   
 5   codigo_uf                                             5571 non-null   str    
 6   siafi_id                                              5571 non-null   int64  
 7   ddd                                                   5571 non-null   str    
 8   fuso_horario                                          5571 non-null  

In [ ]:
# Verifica duplicatas considerando a combinação de Nome e Código da UF
duplicatas_mesmo_estado = df_merge[df_merge.duplicated(subset=['nome', 'codigo_uf'], keep=False)]

if duplicatas_mesmo_estado.empty:
    print("Sucesso! Não existem municípios com nomes repetidos dentro do mesmo state.")
else:
    print(f"Atenção: Foram encontrados {len(duplicatas_mesmo_estado)} duplicatas")
    display(duplicatas_mesmo_estado[['id', 'nome', 'codigo_uf', 'microrregiao.mesorregiao.UF.sigla']].sort_values(by=['nome', 'codigo_uf']))

Sucesso! Não existem municípios com nomes repetidos dentro do mesmo state.


In [ ]:
# Tarefa 3.1c - Compare os nomes de mesorregiões e microrregiões entre as fontes
## Há apenas uma fonte (CSV) que contém as colunas de mesorregião e microrregião, portanto não é possível comparar entre as fontes.



### Tarefa 3.2 - Relatório de problemas de qualidade encontrados

Com base nas análises anteriores, preencha a tabela abaixo:

| # | Fonte | Campo(s) | Dimensão de Qualidade | Problema Encontrado | Frequência | Ação Planejada |
|---|-------|----------|-----------------------|---------------------|------------|----------------|
| 1 | CSV e API | nome | Consistência | Variações de grafia nos nomes dos municípios | 18 municípios | Padronizar nomes usando normalização de texto |
| 2 | API | mesorregiao, microrregiao | Completude | Campos ausentes na fonte API | 100% dos registros | Integrar dados do CSV para preencher |
| 3 | CSV | latitude, longitude | Unicidade | Coordenadas não únicas | 44 duplicatas | Verificar se são erros ou coincidências geográficas |
| 4 | CSV | nome | Unicidade | Nomes de municípios duplicados | 271 duplicatas | Confirmar se são municípios distintos em estados diferentes |
| 5 | CSV | capital | Validade | Valores como inteiros em vez de booleanos | Todos os registros | Converter para tipo booleano |

**Dimensões de qualidade:** Completude, Unicidade, Validade, Consistência, Precisão, Atualidade


### Tarefa 3.3 - Aplicação das Técnicas de Data Wrangling

Para cada problema identificado, aplique a técnica de tratamento mais adequada.  
Use células separadas para cada tipo de tratamento e **adicione um comentário**
explicando a decisão tomada.

**Lembre-se:** Trabalhe em cópias dos DataFrames originais.


In [33]:
# Tarefa 3.3a - Corrija os tipos de dados incorretos em ambos os DataFrames
df_merge.info()

<class 'pandas.DataFrame'>
RangeIndex: 5571 entries, 0 to 5570
Data columns (total 30 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   id                                                    5571 non-null   int64  
 1   nome                                                  5571 non-null   str    
 2   latitude                                              5571 non-null   float64
 3   longitude                                             5571 non-null   float64
 4   capital                                               5571 non-null   bool   
 5   codigo_uf                                             5571 non-null   str    
 6   siafi_id                                              5571 non-null   int64  
 7   ddd                                                   5571 non-null   str    
 8   fuso_horario                                          5571 non-null  

In [36]:
# Tarefa 3.3b - Padronize strings (encoding, case, espaços extras, caracteres especiais)
colunas_categoricas = df_merge.select_dtypes(include=['str'])
for i in colunas_categoricas.columns:
    df_merge[i] = df_merge[i].apply(normalizar)

In [48]:
df_merge.head()

,id,nome,latitude,longitude,capital,codigo_uf,siafi_id,ddd,fuso_horario,microrregiao.id,...,regiao-imediata.nome,regiao-imediata.regiao-intermediaria.id,regiao-imediata.regiao-intermediaria.nome,regiao-imediata.regiao-intermediaria.UF.id,regiao-imediata.regiao-intermediaria.UF.sigla,regiao-imediata.regiao-intermediaria.UF.nome,regiao-imediata.regiao-intermediaria.UF.regiao.id,regiao-imediata.regiao-intermediaria.UF.regiao.sigla,regiao-imediata.regiao-intermediaria.UF.regiao.nome,microrregiao
0,5200050,abadia de goias,-16.75730,-49.4412,False,52,1050,62,america/sao_paulo,52010.0,...,goiania,5201,goiania,52,go,goias,5,co,centro-oeste,NaN
1,3100104,abadia dos dourados,-18.48310,-47.3916,False,31,4001,34,america/sao_paulo,31019.0,...,monte carmelo,3111,uberlandia,31,mg,minas gerais,3,se,sudeste,NaN
2,5200100,abadiania,-16.19700,-48.7057,False,52,9201,62,america/sao_paulo,52012.0,...,anapolis,5201,goiania,52,go,goias,5,co,centro-oeste,NaN
3,3100203,abaete,-19.15510,-45.4444,False,31,4003,37,america/sao_paulo,31024.0,...,abaete,3113,divinopolis,31,mg,minas gerais,3,se,sudeste,NaN
4,1500107,abaetetuba,-1.72183,-48.8788,False,15,401,91,america/sao_paulo,15011.0,...,abaetetuba,1501,belem,15,pa,para,1,n,norte,NaN


In [ ]:
# Tarefa 3.3c - Trate os valores nulos encontrados
df_merge.isnull().sum()
df_merge

municipio_nulo = df_merge[df_merge['microrregiao.id'].isnull()]

display(municipio_nulo[['id', 'nome']])

,id,nome
5570,5101837,boa esperanca do norte


In [50]:
# Dicionário com os valores geográficos mapeados (herdados de Sorriso/MT)
valores_preenchimento = {
    'microrregiao.id': 51013.0,
    'microrregiao.nome': 'alto teles pires', 
    'microrregiao.mesorregiao.id': 5103.0,
    'microrregiao.mesorregiao.nome': 'norte mato-grossense',
    'microrregiao.mesorregiao.UF.id': 51.0,
    'microrregiao.mesorregiao.UF.sigla': 'mt',
    'microrregiao.mesorregiao.UF.nome': 'mato grosso',
    'microrregiao.mesorregiao.UF.regiao.id': 5.0,
    'microrregiao.mesorregiao.UF.regiao.sigla': 'co',
    'microrregiao.mesorregiao.UF.regiao.nome': 'centro-oeste'
}

for coluna, valor in valores_preenchimento.items():
    df_merge.loc[df_merge['id'] == 5101837, coluna] = valor

# Validação: Verifica se ainda sobrou algum valor nulo no DataFrame inteiro
nulos_restantes = df_merge.isnull().sum().max()
df_merge = df_merge.drop(columns=["microrregiao"])
print(f"Tratamento concluído. Quantidade máxima de nulos nas colunas agora: {nulos_restantes}")

Tratamento concluído. Quantidade máxima de nulos nas colunas agora: 5571


In [51]:
df_merge.isnull().sum()

id                                                      0
nome                                                    0
latitude                                                0
longitude                                               0
capital                                                 0
codigo_uf                                               0
siafi_id                                                0
ddd                                                     0
fuso_horario                                            0
microrregiao.id                                         0
microrregiao.nome                                       0
microrregiao.mesorregiao.id                             0
microrregiao.mesorregiao.nome                           0
microrregiao.mesorregiao.UF.id                          0
microrregiao.mesorregiao.UF.sigla                       0
microrregiao.mesorregiao.UF.nome                        0
microrregiao.mesorregiao.UF.regiao.id                   0
microrregiao.m

In [57]:
# Tarefa 3.3d - Valide as coordenadas geográficas do CSV
# Latitude válida: entre -90 e 90
# Longitude válida: entre -180 e 180
# Para o Brasil: lat entre -34 e 6; lon entre -74 e -28
for i in df_merge[(df_merge['latitude'] < -34) | (df_merge['latitude'] > 6)]:
    latitude_invalida = latitude_invalida.append(df_merge.loc[i])
for i in df_merge[(df_merge['longitude'] < -74) | (df_merge['longitude'] > -28)]:
    longitude_invalida = longitude_invalida.append(df_merge.loc[i])
print(longitude_invalida)
print(latitude_invalida)

AttributeError: 'DataFrame' object has no attribute 'append'

In [ ]:
# Tarefa 3.3e - Aplique demais correções identificadas no diagnóstico


---
# Parte 4 - Integração das Fontes e Validação Final

### Tarefa 4.1 - Preparar chaves de integração

Para unir as duas fontes, você precisa de uma **chave de integração comum**.  
O CSV usa o código IBGE com 7 dígitos. A API retorna o campo `id_municipio`.

Verifique se os dois campos têm o mesmo formato e, se necessário, ajuste
um deles para garantir que o join funcionará corretamente.

> **Atenção:** Verifique o número de dígitos. O código IBGE completo tem 7 dígitos.
> Se a API retornar 6, pode ser necessário adicionar o dígito verificador ou vice-versa.


In [ ]:
# Tarefa 4.1 - Verifique e ajuste as chaves de integração


### Tarefa 4.2 - Integrar as duas fontes

Realize o merge entre os dois DataFrames tratados usando a chave de integração.  

O DataFrame integrado final deve conter, para cada município:

| Coluna | Origem |
|--------|--------|
| `codigo_ibge` | Chave de integração |
| `nome_municipio` | API (ou CSV — escolha e justifique) |
| `latitude` | CSV |
| `longitude` | CSV |
| `capital` | CSV |
| `sigla_uf` | API |
| `nome_uf` | API |
| `sigla_regiao` | API |
| `nome_regiao` | API |
| `nome_mesorregiao` | API (ou CSV — justifique a escolha) |
| `nome_microrregiao` | API (ou CSV — justifique a escolha) |

> **Dica:** Use `pd.merge()` com `how='inner'` para início.  
> Depois verifique quantos registros foram perdidos e investigue os casos.


In [ ]:
# Tarefa 4.2 - Realize o merge das duas fontes e construa o DataFrame integrado


### Tarefa 4.3 - Validação final do dataset integrado

Antes de declarar o dataset como pronto para carga, valide:

1. O número de municípios está correto (esperado: 5.570)?
2. Nenhum campo obrigatório possui nulos?
3. Os valores de `sigla_uf` cobrem todos os 27 estados?
4. Os valores de `sigla_regiao` cobrem exatamente 5 regiões (N, NE, CO, SE, S)?
5. As coordenadas de todos os municípios estão dentro dos limites geográficos do Brasil?
6. O campo `capital` possui exatamente 27 municípios marcados como capital (1 por UF)?

Escreva **assertions** (verificações programáticas) para cada item acima.


In [ ]:
# Tarefa 4.3 - Escreva verificações de qualidade para o dataset integrado


---
# Parte 5 - Análises Exploratórias e Reflexão

### Tarefa 5.1 - Análises com o dataset integrado

Responda às perguntas abaixo usando o dataset integrado e tratado:


**Pergunta A:** Quantos municípios existem em cada região do Brasil?
Qual região concentra o maior número de municípios?


In [ ]:
# Pergunta A


**Pergunta B:** Qual estado possui o maior número de mesorregiões?
E o maior número de microrregiões?


In [ ]:
# Pergunta B


**Pergunta C:** Identifique o município mais ao norte, mais ao sul, mais a leste
e mais a oeste do Brasil usando as coordenadas geográficas.


In [ ]:
# Pergunta C


**Pergunta D:** Calcule o centroide geográfico (latitude e longitude médias)
de cada região do Brasil. Qual região tem o centroide mais próximo do equador?


In [ ]:
# Pergunta D


---
### Tarefa 5.2 - Reflexão sobre o processo

Responda às perguntas abaixo com base na sua experiência ao longo desta atividade:

**1. Qual das duas fontes apresentou maior número de problemas de qualidade?
Que fatores podem explicar essa diferença?**

*Sua resposta:*

---

**2. Qual técnica de Data Wrangling você considerou mais desafiadora de implementar e por quê?**

*Sua resposta:*

---

**3. Ao fazer o merge entre as duas fontes, alguns municípios podem não ter encontrado
correspondência. O que você faria com esses registros em um projeto real de DW?
Descartaria ou investigaria? Como documentaria essa decisão?**

*Sua resposta:*

---

**4. Se você precisasse atualizar este dataset mensalmente (novos municípios criados,
renomeações, mudanças de divisão administrativa), qual estratégia de extração
(full load, incremental ou CDC) seria mais adequada para cada fonte? Justifique.**

*Sua resposta:*

---

**5. Este dataset de municípios poderia ser usado como uma dimensão em um DW?
Se sim, qual seria o nome dessa dimensão, quais colunas ela incluiria,
e quais tabelas fato poderiam se relacionar com ela?**

*Sua resposta:*
